# ⚽ TAJ — Football Tactical Intelligence · Colab Edition
**Open-data, evidence-led spatial analysis** — قابل اجرا روی Google Colab بدون GPU.

> داده‌های Demo کاملاً **مصنوعی** هستند. ادعاهای تاکتیکی فقط زمانی معتبرتر می‌شوند که روی Tracking واقعی و با شواهد مستقل بررسی شوند. این نوت‌بوک هیچ پیش‌بینی نتیجه مسابقه ارائه نمی‌کند.

[Repository](https://github.com/prestigegitserp/Taj_soccer_recommender) · [Static Visual Lab](https://prestigegitserp.github.io/Taj_soccer_recommender/) (پس از فعال‌سازی Pages)

**مراحل:** نصب → Demo → Heatmap → گزارش و پیشنهاد → IDSSE واقعی → SkillCorner واقعی.

## 1 · Install dependencies and fetch your repository
این سلول را یک بار اجرا کن. نصب به اینترنت Colab نیاز دارد.

In [ ]:
import os, sys, subprocess
os.chdir("/content")
if not os.path.isdir("/content/Taj_soccer_recommender"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/prestigegitserp/Taj_soccer_recommender.git"],check=True)
os.chdir("/content/Taj_soccer_recommender")
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".[all]","ipywidgets"],check=True)
print("✅ Workspace:",os.getcwd())

## 2 · Smoke-test with **synthetic** data
این داده برای آزمایش کد و رابط کاربری است و نباید به‌عنوان نتیجه بازی واقعی گزارش شود.

In [ ]:
from taj.analysis import create_demo, build_report, export_report
from taj.plotting import pitch_figure
from taj.schema import at_frame,validate_tracking
from IPython.display import display, Markdown
import pandas as pd
frames, ball, report = create_demo("data/demo")
display(Markdown("**Rows:** "+str(len(frames))+" · **Synthetic:** "+str(report["synthetic"])))
display(report["quality"])

## 3 · Interactive tactical pitch
اسلایدر را حرکت بده و تیک Heatmap را تغییر بده؛ مدل شاخص دسترسی فضایی است، نه احتمال کالیبره‌شده.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
indices = frames[["period","frame_id","t_s"]].drop_duplicates().reset_index(drop=True)
slider=widgets.IntSlider(value=len(indices)//2,min=0,max=len(indices)-1,step=1,
                         description="Frame",continuous_update=False,layout=widgets.Layout(width="70%"))
heat=widgets.Checkbox(value=True,description="Pitch access")
out=widgets.Output()
def render(_=None):
    with out:
        clear_output(wait=True)
        k=indices.iloc[slider.value]
        f=at_frame(frames,int(k.period),str(k.frame_id))
        b=ball[ball.frame_id==str(k.frame_id)]
        display(pitch_figure(f,b,heatmap=heat.value,
                    title="SYNTHETIC TEST · %.1f s"%k.t_s))
slider.observe(render,names="value")
heat.observe(render,names="value")
display(widgets.VBox([widgets.HBox([slider,heat]),out]))
render()

## 4 · Auditable tactical findings
پیشنهاد فقط زمانی ساخته می‌شود که دست‌کم چند پنجره زمانی مستقل از فرضیه پشتیبانی کنند.

In [ ]:
report=build_report(frames,ball,defending="Home",goal=1,stride=3,min_windows=2)
display(Markdown("### Hypotheses ("+str(len(report["findings"]))+")"))
for f in report["findings"]:
    display(Markdown("**"+f["statement_fa"]+"** · "+str(f["n_independent_windows"])+" windows"))
display(Markdown("### Suggestions ("+str(len(report["recommendations"]))+")"))
for r in report["recommendations"]:
    display(Markdown("- **"+r["title_fa"]+"** · "+r["risk_fa"]))
p=export_report(report,"outputs/demo_report.json")
print("Report saved:",p)

## 5 · Real IDSSE data
**اجرا اختیاری**: متغیر RUN_IDSSE را True کن؛ دانلود فایل‌های واقعی ممکن است چند دقیقه و حافظه بیشتر لازم داشته باشد. ID های مجاز در `taj.ingest.IDSSE_IDS` قرار دارند.

In [ ]:
RUN_IDSSE = False
if RUN_IDSSE:
    from taj.ingest import load_idsse,write_bundle
    tracking, real_ball, real_events = load_idsse(
        match_id="J03WMX",limit=1200,sample_rate=2,with_events=False)
    stats = write_bundle(tracking,real_ball,real_events,"data/idsse_J03WMX")
    print(stats)
    r=build_report(tracking,real_ball,real_events,stride=10,min_windows=4)
    print("Real-data hypotheses:",len(r["findings"]))
    display(pitch_figure(at_frame(tracking,int(tracking.period.iloc[0]),
                           str(tracking.frame_id.iloc[0])),title="IDSSE / real tracking"))
else:
    print("IDSSE download skipped. Set RUN_IDSSE = True when ready.")

## 6 · Real SkillCorner via Git LFS
**اجرا اختیاری**: Match 1925299 شامل داده‌های حجیم Git LFS است. در Colab فایل‌های واقعی لازم‌اند؛ Pointer صدبایتی فایل داده نیست. `is_detected` جداگانه ذخیره می‌شود.

In [ ]:
RUN_SKILLCORNER = False
if RUN_SKILLCORNER:
    import shutil
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","git-lfs"],check=True)
    subprocess.run(["git","lfs","install"],check=True)
    if not os.path.isdir("external/opendata"):
        env={**os.environ,"GIT_LFS_SKIP_SMUDGE":"1"}
        subprocess.run(["git","clone","--depth","1","https://github.com/SkillCorner/opendata.git",
                       "external/opendata"],env=env,check=True)
    subprocess.run(["git","lfs","pull","-I",
                    "data/matches/1925299/1925299_tracking_extrapolated.jsonl"],
                   cwd="external/opendata",check=True)
    from taj.ingest import load_skillcorner_local, write_bundle
    sc,bsc,esc=load_skillcorner_local("1925299","external/opendata",
                                     limit=2000,stride=10)
    print(write_bundle(sc,bsc,esc,"data/skillcorner_1925299"))
else:
    print("SkillCorner LFS download skipped. Set RUN_SKILLCORNER = True.")

## 7 · Export to Google Drive (optional)
در صورت نیاز فایل‌های Parquet و گزارش JSON را به Drive کپی کن. محتوای خام شامل داده ویدیویی نیست.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    from pathlib import Path
    import shutil
    drive.mount("/content/drive")
    dest=Path("/content/drive/MyDrive/TajSoccer")
    dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2("outputs/demo_report.json",dest/"demo_report.json")
    print("Saved:",dest)
else:
    print("Local outputs live in /content/Taj_soccer_recommender/outputs")

## Scientific and legal limitations
- IDSSE: tracking + comprehensive event feed, CC BY 4.0.
- SkillCorner: partial broadcast coverage, extrapolated points (`is_detected=False`), dynamic events **not equivalent** to a complete event feed.
- Do not infer a team weakness from a single frame, or interpret heuristic scores as goal probabilities.
- For real scouting, inspect video, derive period-correct directions, and validate on matches held out in time.

See `docs/METHODOLOGY.md` and `README.md` for limitations and next steps.

## 8 · Advanced spatial features on the chosen match
Choose any processed bundle. For SkillCorner, verify goal directions *per period*; the notebook refuses to guess automatically.

In [ ]:
from pathlib import Path
from taj.ingest import read_bundle
from taj.schema import at_frame
from taj.advanced import SpatialConfig, quick_report_frame
from taj.features import spatial_summary
import ipywidgets as widgets
from IPython.display import display

paths = sorted(str(p) for p in Path("data").iterdir()
               if p.is_dir() and (p/"tracking.parquet").exists())
dataset_picker = widgets.Dropdown(options=paths, description="Dataset",
                                  layout=widgets.Layout(width="70%"))
display(dataset_picker)
print("Pick a dataset here, then run the next cell.")


In [ ]:
from taj.analysis import build_report
from taj.plotting import pitch_figure
from taj.events import event_summary
from taj.schema import validate_tracking
from IPython.display import display
tracking, bb, ee = read_bundle(dataset_picker.value)
quality = validate_tracking(tracking)
print("Quality:", quality, "Event summary:", event_summary(ee))
sample = tracking[["period","frame_id","t_s"]].drop_duplicates().sort_values(["period","t_s"])
sample = sample.iloc[len(sample)//2]
f = at_frame(tracking, int(sample.period), str(sample.frame_id))
chosen_team = "Home"
periods = sorted(int(p) for p in tracking.period.unique())
# IDSSE uses static-home-away orientation: Home defends -x in every period.
# SkillCorner raw tracking direction needs manual video/metadata verification.
if tracking.provider.iloc[0] in ("idsse", "synthetic"):
    period_goals = {p:-1 for p in periods}
else:
    period_goals = {1:-1, 2:1}  # EXAMPLE ONLY - VERIFY BEFORE RESEARCH OUTPUT.
    if set(period_goals) != set(periods):
        raise ValueError("Supply defended-goal signs for every period")
result = quick_report_frame(f,defending=chosen_team,
                            goal=period_goals[int(sample.period)],
                            cfg=SpatialConfig(grid_m=4))
display(result)
display(pitch_figure(f,title="Data from "+str(dataset_picker.value),heatmap=True))


## 9 · Evidence-gated full analysis
The advanced evidence model uses sampled tracking frames and abstains from recommendations if too few separated time windows support a pattern. A team weakness is not proven merely by empty space.

In [ ]:
from taj.recommender import recommend
from taj.analysis import build_report, export_report
from taj.export import export_html
from pathlib import Path
frame_count = tracking[["period","frame_id"]].drop_duplicates().shape[0]
stride = max(1,(frame_count+199)//200)  # bounded Colab work
analysis_report = build_report(tracking,bb,ee,defending=chosen_team,
                               direction_by_period=period_goals,
                               stride=stride,min_windows=4)
print("Processed frames:",analysis_report["processed_frames"])
print("Exposure hypotheses:",len(analysis_report["findings"]))
print("Suggestions:",len(analysis_report["recommendations"]))
for warning in analysis_report["warnings"]:
    print("WARNING:",warning)
display(analysis_report["findings"][:3])
display(analysis_report["recommendations"][:3])
Path("outputs").mkdir(exist_ok=True)
export_report(analysis_report,"outputs/analysis_report.json")
export_html(analysis_report,tracking,bb,"outputs/analysis_report.html")
print("Saved outputs/analysis_report.json and outputs/analysis_report.html")


## 10 · Browser Intelligent UI export
Generate an interactive JSON bundle containing **source player coordinates** and computed heatmaps, then open docs/viewer.html on GitHub Pages and upload the JSON. File is processed locally in your browser.

In [ ]:
from taj.viewer import build_viewer_payload,export_viewer_json
viewer = build_viewer_payload(tracking,bb,analysis_report,
                              max_frames=150,step_m=6)
export_viewer_json(viewer,"outputs/viewer.json")
print("Exported browser frames:",len(viewer["frames"]))
print("Viewer: https://prestigegitserp.github.io/Taj_soccer_recommender/viewer.html")
print("On Colab, use files.download('outputs/viewer.json') to download the bundle.")


## 11 · What-if: move a player and recompute spatial access
This is a **geometric sensitivity test**, not a simulation of how the whole opposing team will react. Do not interpret a positive differential as causal game-winning impact.

In [ ]:
from taj.plotting import control_difference
from taj.advanced import SpatialConfig
import numpy as np
player_choice = f[f.team=="Home"].iloc[1]
moved = f.copy()
idx = moved.index[moved.player_id==player_choice.player_id][0]
moved.loc[idx,"x"] = np.clip(float(moved.loc[idx,"x"])+8,-52.5,52.5)
moved.loc[idx,"y"] = np.clip(float(moved.loc[idx,"y"])+4,-34,34)
display(control_difference(f,moved,SpatialConfig(grid_m=4)))


## 12 · Optional pre-match baseline from YOUR historical CSV
Required columns: date, home_team, away_team, home_goals, away_goals. Uses earlier games only and does not include tactical tracking, lineups or injuries. Do not invent probabilities without a valid results file.

In [ ]:
PREDICT_FROM_CSV = False
if PREDICT_FROM_CSV:
    from google.colab import files
    from taj.prediction import predict,walk_forward_brier
    import io, pandas as pd
    incoming = files.upload()
    first_name = next(iter(incoming))
    results = pd.read_csv(io.BytesIO(incoming[first_name]))
    print("Columns:",results.columns.tolist())
    home_team = input("Exact home-team name in CSV: ")
    away_team = input("Exact away-team name in CSV: ")
    as_of = input("Match date YYYY-MM-DD: ")
    display(predict(results,home_team,away_team,as_of))
    display(walk_forward_brier(results,min_games=50))
else:
    print("No verified historical results supplied. Prediction intentionally disabled.")


## 13 · Download artifacts to your machine
The notebook keeps model reports, portable HTML and the real-data web replay bundle in outputs/ (not committed to GitHub).

In [ ]:
DOWNLOAD_ARTIFACTS = False
if DOWNLOAD_ARTIFACTS:
    from google.colab import files
    for filename in ["outputs/analysis_report.json",
                     "outputs/analysis_report.html",
                     "outputs/viewer.json"]:
        files.download(filename)
else:
    print("Set DOWNLOAD_ARTIFACTS=True after successful real-data processing.")
